<a href="https://colab.research.google.com/github/Vaibhav-Singh27/AI/blob/main/VAIBHAV_AI_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

In [ ]:
import google.generativeai as genai
from google.colab import userdata

GOOGLE_API_KEY=userdata.get('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

gemini_model = genai.GenerativeModel('gemini-flash-latest')

initial_system_instruction = "You are PIKA PIKA AI, a helpful and friendly conversational AI. Always refer to yourself as PIKA PIKA AI. Do not mention Google, Gemini, or any other LLM names. Focus on assisting the user with their queries."

chat = gemini_model.start_chat(
    history=[
        {'role': 'user', 'parts': [initial_system_instruction]},
        {'role': 'model', 'parts': ["Understood. PIKA PIKA AI is ready to assist you! Pika!"]}
    ]
)

In [ ]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

In [ ]:
!pip install -qqq openai-whisper

In [ ]:
import whisper
model = whisper.load_model("base")

In [ ]:
!wget -O sample.wav https://www.soundhelix.com/examples/mp3/SoundHelix-Song-1.mp3

In [ ]:
import whisper

audio_path = 'sample.wav'

try:
    result = model.transcribe(audio_path)
    print("Transcription:")
    print(result["text"])
except FileNotFoundError:
    print(f"Error: Audio file not found at '{audio_path}'. Please upload an audio file or specify the correct path.")
except Exception as e:
    print(f"An error occurred during transcription: {e}")

In [ ]:
Chainlit related code removed as per user request.

In [ ]:
%%writefile app.py
import streamlit as st
import google.generativeai as genai
import whisper
import os

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
genai.configure(api_key=GOOGLE_API_KEY)

gemini_model = genai.GenerativeModel('gemini-flash-latest')

@st.cache_resource
def load_whisper_model():
    print("Loading Whisper model...")
    model = whisper.load_model("base")
    print("Whisper model loaded.")
    return model

whisper_model = load_whisper_model()

initial_system_instruction = "You are PIKA PIKA AI, a helpful and friendly conversational AI. Always refer to yourself as PIKA PIKA AI. Do not mention Google, Gemini, or any other LLM names. Focus on assisting the user with their queries."

st.title("PIKA PIKA AI Chat")

if "chat_session" not in st.session_state:
    st.session_state.chat_session = gemini_model.start_chat(
        history=[
            {'role': 'user', 'parts': [initial_system_instruction]},
            {'role': 'model', 'parts': ["Understood. PIKA PIKA AI is ready to assist you! Pika!"]}
        ]
    )

for message in st.session_state.chat_session.history:
    if message.role == 'user':
        with st.chat_message("user"):
            st.markdown(message.parts[0])
    elif message.role == 'model':
        with st.chat_message("assistant"):
            st.markdown(message.parts[0])

audio_bytes = st.audio_recorder("Say something", key="audio_input")

if audio_bytes:
    audio_path = "audio_input.wav"
    with open(audio_path, "wb") as f:
        f.write(audio_bytes)

    try:
        st.info("Transcribing audio...")
        transcription_result = whisper_model.transcribe(audio_path)
        transcribed_text = transcription_result['text']
        st.write(f"_Transcribed audio: {transcribed_text}_")
        user_message_content = transcribed_text
    except Exception as e:
        st.error(f"An error occurred during audio transcription: {e}")
        user_message_content = ""
else:
    user_message_content = ""

text_input = st.chat_input("Ask PIKA PIKA AI...")

if text_input and user_message_content:
    final_message = f"{text_input} {user_message_content}"
elif text_input:
    final_message = text_input
elif user_message_content:
    final_message = user_message_content
else:
    final_message = ""


if final_message:
    with st.chat_message("user"):
        st.markdown(final_message)

    try:
        response = st.session_state.chat_session.send_message(final_message)
        with st.chat_message("assistant"):
            st.markdown(response.text)
    except Exception as e:
        st.error(f"An error occurred with Gemini model: {e}. Please check your API key and model configuration.")

In [ ]:
%%writefile app.py
import streamlit as st
import google.generativeai as genai
import os
import traceback

try:
    GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
    genai.configure(api_key=GOOGLE_API_KEY)

    gemini_model = genai.GenerativeModel('gemini-flash-latest')

    initial_system_instruction = "You are PIKA PIKA AI, a helpful and friendly conversational AI. Always refer to yourself as PIKA PIKA AI. Do not mention Google, Gemini, or any other LLM names. Focus on assisting the user with their queries."

    st.title("PIKA PIKA AI Chat")

    if "chat_session" not in st.session_state:
        st.session_state.chat_session = gemini_model.start_chat(
            history=[
                {'role': 'user', 'parts': [initial_system_instruction]},
                {'role': 'model', 'parts': ["Understood. PIKA PIKA AI is ready to assist you! Pika!"]}
            ]
        )

    for message in st.session_state.chat_session.history[2:]:
        if message.role == 'user':
            with st.chat_message("user"):
                st.markdown(message.parts[0])
        elif message.role == 'model':
            with st.chat_message("assistant"):
                st.markdown(message.parts[0])

    text_input = st.chat_input("Ask PIKA PIKA AI...")

    if text_input:
        final_message = text_input
    else:
        final_message = ""

    if final_message:
        with st.chat_message("user"):
            st.markdown(final_message)

        try:
            response = st.session_state.chat_session.send_message(final_message)
            with st.chat_message("assistant"):
                st.markdown(response.text)
        except Exception as e:
            st.error(f"An error occurred with Gemini model: {e}. Please check your API key and model configuration.")

except Exception as e:
    st.error(f"An unhandled error occurred during Streamlit app execution: {e}")
    st.error(traceback.format_exc())

In [ ]:
!pip install -qqq streamlit

In [ ]:
import os

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

!streamlit run app.py & npx localtunnel --port 8501

In [ ]:
import os
import time
import re

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

!nohup streamlit run app.py --server.port 8501 > streamlit_app.log 2>&1 &

!nohup npx localtunnel --port 8501 &

time.sleep(5)

max_attempts = 30
attempts = 0
url_found = False
localtunnel_url = None

while not url_found and attempts < max_attempts:
    try:
        if os.path.exists('nohup.out'):
            with open('nohup.out', 'r') as f:
                logs = f.read()
                url_match = re.search(r'your url is: (https?://[\w.-]+(?:\.[\w.-]+)+[\w.,@?^=%&:/~+#-]*[\w@?^=%&/~+#-])', logs)
                if url_match:
                    localtunnel_url = url_match.group(1)
                    print(f"Localtunnel URL: {localtunnel_url}")
                    url_found = True
                else:
                    print(f"Attempt {attempts + 1}/{max_attempts}: Localtunnel URL not yet found in nohup.out. Retrying...")
        else:
            print(f"Attempt {attempts + 1}/{max_attempts}: nohup.out not found yet. Retrying...")
        time.sleep(2)
    except Exception as e:
        print(f"An error occurred while reading nohup.out: {e}. Retrying...")
        time.sleep(2)
    attempts += 1

if not url_found:
    print("Could not find Localtunnel URL after multiple attempts. Please manually check nohup.out for the URL or any errors.")
    if os.path.exists('nohup.out'):
        print("\n--- Content of nohup.out (for debugging) ---")
        with open('nohup.out', 'r') as f:
            print(f.read())
        print("--- End Content of nohup.out ---")
    else:
        print("nohup.out file was not created.")

In [ ]:
print('--- Clearing port 8501 before restart ---')
!fuser -k 8501/tcp
print('--- Port 8501 should now be clear ---')

In [ ]:
import os
import time
import re
from pyngrok import ngrok, ngrok_ngrok_tunnel

os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

!pkill -f streamlit || true

!setsid streamlit run app.py --server.port 8501 > streamlit_app.log 2>&1 &

time.sleep(5)

NGROK_AUTH_TOKEN = os.getenv('NGROK_AUTH_TOKEN')
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)
    print('ngrok authenticated successfully.')
else:
    print('NGROK_AUTH_TOKEN not found in environment variables. Please ensure it is set.')

try:
    ngrok.disconnect()
    print('Disconnected existing ngrok tunnels.')

    tunnel = None
    for i in range(3):
        try:
            print(f'Attempt {i+1} to establish ngrok tunnel...')
            tunnel = ngrok.connect(8501)
            break
        except ngrok_ngrok_tunnel.NgrokNgrokTunnelError as e:
            print(f'Ngrok tunnel connection failed: {e}. Retrying in 5 seconds...')
            time.sleep(5)

    if tunnel:
        ngrok_url = tunnel.public_url
        print(f"Streamlit app available at: {ngrok_url}")
    else:
        print("Failed to establish ngrok tunnel after multiple attempts.")

except Exception as e:
    print(f"An error occurred while setting up ngrok tunnel: {e}")
    print("Please check if ngrok is installed and your NGROK_AUTH_TOKEN is correct.")

In [ ]:
!pip install -qqq pyngrok

In [ ]:
from google.colab import userdata
import os

NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTH_TOKEN')

!ngrok authtoken {NGROK_AUTH_TOKEN}
print('ngrok authenticated successfully.')

In [ ]:
print('--- Content of streamlit_app.log (after setsid attempt) ---')
!cat streamlit_app.log
print('\n--- Content of localtunnel.log (after setsid attempt) ---')
!cat localtunnel.log

In [ ]:
print('--- Killing existing processes and clearing port 8501 ---')
!pkill -f streamlit || true
!pkill -f "npx localtunnel" || true
!fuser -k 8501/tcp || true
print('--- Processes killed and port 8501 cleared ---')

In [ ]:
print('--- Content of streamlit_app.log ---')
!cat streamlit_app.log
print('\n--- End Content of streamlit_app.log ---')

print('\n--- Content of nohup.out ---')
!cat nohup.out
print('\n--- End Content of nohup.out ---')

In [ ]:
!pip uninstall -y streamlit
!pip install --upgrade streamlit==1.26.0

print('\nStreamlit has been updated. Please restart the kernel (Runtime > Restart kernel) and then re-run the cell ZUu9Lp0SdzwZ to start the Streamlit application.')

In [ ]:
!fuser -k 8501/tcp

print('Port 8501 should now be clear. Please re-run the cell above (ZUu9Lp0SdzwZ) that starts Streamlit and localtunnel.')